# W4 Quantization Evaluation Analysis

This notebook inspects the first 4-way evaluation for the project question:

> Does the GRPO post-training gain survive 4-bit quantization?

It compares the same GSM8K test split across four variants:

| Variant | Precision | Meaning |
| --- | --- | --- |
| `base_fp16` | FP16/BF16 runtime | original Qwen2.5-0.5B-Instruct |
| `g8_dr100_fp16` | FP16/BF16 runtime | Day 3 GRPO checkpoint |
| `base_w4` | bitsandbytes NF4 W4 | base loaded in 4-bit |
| `g8_dr100_w4` | bitsandbytes NF4 W4 | GRPO checkpoint loaded in 4-bit |

Important: `W4` here means **bitsandbytes NF4 double-quant load-time 4-bit**, not AWQ W4G128. This is a real 4-bit inference path, but it is not the final AWQ result.

## How To Read This Notebook

The headline metric is not one raw accuracy number. The core comparison is:

- `delta_fp16 = accuracy(g8_dr100_fp16) - accuracy(base_fp16)`
- `delta_w4 = accuracy(g8_dr100_w4) - accuracy(base_w4)`
- `gain_survival = delta_w4 - delta_fp16`

If `gain_survival` is near zero, the GRPO gain survived W4. If it is strongly negative, W4 erased or reversed the GRPO benefit.

This run uses only 50 GSM8K examples, so treat it as a strong smoke result, not a final paper-quality result. The notebook reports 1-sigma binomial standard error to make that uncertainty visible.

## Configuration

The defaults point at the completed Great Lakes run `pqs-eval-w4-test50` / job `51161262`.

In [ ]:
from pathlib import Path
import json
import math

from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

PQS_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/pqs')
EVAL_DIR = PQS_ROOT / 'evals/gsm8k_compare_test50_g8_dr100_bnb_w4_final'
OUTLIER_DIR = PQS_ROOT / 'diagnostics/weight_outliers_0p5b_g8_dr100'
LOG_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/repos/posttrain-quant-serve/logs')
JOB_ID = '51161262'

RESULTS_MATRIX = EVAL_DIR / 'results_matrix.csv'
SUMMARY_JSON = EVAL_DIR / 'summary.json'
PAIRED_QUANT = EVAL_DIR / 'paired_quant_effect.csv'

plt.style.use('default')
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 160)

print('EVAL_DIR =', EVAL_DIR)
print('SUMMARY_JSON exists:', SUMMARY_JSON.exists())
print('RESULTS_MATRIX exists:', RESULTS_MATRIX.exists())

## Helper Functions

These helpers keep missing artifacts from crashing the notebook. If a file is absent, the notebook says so directly.

In [ ]:
def safe_display(df: pd.DataFrame | None, message: str = 'No rows to display.'):
    if df is None or df.empty:
        print(message)
    else:
        display(df)


def read_json(path: Path) -> dict:
    if not path.exists():
        print(f'Missing JSON: {path}')
        return {}
    return json.loads(path.read_text())


def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f'Missing JSONL: {path}')
        return pd.DataFrame()
    return pd.DataFrame(json.loads(line) for line in path.read_text().splitlines() if line.strip())


def fmt_pct(x):
    return None if pd.isna(x) else f'{100*x:.1f}%'


def one_sigma_interval(acc: float, stderr: float | None) -> tuple[float | None, float | None]:
    if stderr is None or pd.isna(stderr):
        return None, None
    return max(0.0, acc - stderr), min(1.0, acc + stderr)


def classify_quant_change(row):
    if bool(row.get('fp16_correct')) is False and bool(row.get('w4_correct')) is True:
        return 'improved'
    if bool(row.get('fp16_correct')) is True and bool(row.get('w4_correct')) is False:
        return 'worsened'
    return 'unchanged'

## 1. Artifact Checklist

This section verifies that the Slurm job completed and that the expected result files exist.

In [ ]:
expected = [
    SUMMARY_JSON,
    RESULTS_MATRIX,
    PAIRED_QUANT,
    EVAL_DIR / 'base_fp16_predictions.jsonl',
    EVAL_DIR / 'g8_dr100_fp16_predictions.jsonl',
    EVAL_DIR / 'base_w4_predictions.jsonl',
    EVAL_DIR / 'g8_dr100_w4_predictions.jsonl',
]

artifact_rows = []
for path in expected:
    artifact_rows.append({
        'path': str(path),
        'exists': path.exists(),
        'size_kib': round(path.stat().st_size / 1024, 1) if path.exists() else None,
    })
artifact_df = pd.DataFrame(artifact_rows)
safe_display(artifact_df)

log_rows = []
for suffix in ['out', 'err']:
    path = LOG_DIR / f'pqs-eval-w4-test50-{JOB_ID}.{suffix}'
    log_rows.append({
        'log': str(path),
        'exists': path.exists(),
        'size_kib': round(path.stat().st_size / 1024, 1) if path.exists() else None,
    })
safe_display(pd.DataFrame(log_rows))

## 2. Scorecard

The scorecard is the main result table. Accuracy is exact match on the parsed final answer. PPL is teacher-forced likelihood of the official GSM8K reference solution; lower is better.

In [ ]:
summary = read_json(SUMMARY_JSON)
matrix = pd.read_csv(RESULTS_MATRIX) if RESULTS_MATRIX.exists() else pd.DataFrame()

if not matrix.empty:
    matrix['accuracy_pct'] = matrix['accuracy'].map(fmt_pct)
    matrix['stderr_pct'] = matrix['accuracy_stderr'].map(fmt_pct)
    intervals = matrix.apply(lambda r: one_sigma_interval(r['accuracy'], r['accuracy_stderr']), axis=1)
    matrix['acc_1sigma_low'] = [lo for lo, _ in intervals]
    matrix['acc_1sigma_high'] = [hi for _, hi in intervals]

score_cols = [
    'variant', 'precision', 'correct', 'num_examples', 'accuracy', 'accuracy_stderr',
    'parse_rate', 'prompt_leak_rate', 'reference_ppl', 'completion_chars_mean'
]
safe_display(matrix[score_cols] if not matrix.empty else matrix)

headline = {
    'delta_fp16': summary.get('delta_fp16'),
    'delta_w4': summary.get('delta_w4'),
    'gain_survival': summary.get('gain_survival'),
    'quant_drop_base': summary.get('quant_drop_base'),
    'quant_drop_g8': summary.get('quant_drop_g8'),
}
display(pd.DataFrame([headline]))

notes = summary.get('notes') or []
for note in notes:
    display(Markdown(f'**Note:** {note}'))

## 3. Headline Interpretation

This cell turns the scorecard into the research statement. It intentionally uses the saved `summary.json` numbers rather than recomputing hidden values.

In [ ]:
delta_fp16 = summary.get('delta_fp16')
delta_w4 = summary.get('delta_w4')
gain_survival = summary.get('gain_survival')
quant_drop_base = summary.get('quant_drop_base')
quant_drop_g8 = summary.get('quant_drop_g8')

if delta_fp16 is None or delta_w4 is None or gain_survival is None:
    display(Markdown('**Incomplete result:** both FP16 and W4 rows are required to measure gain survival.'))
else:
    verdict = 'survives' if gain_survival >= -0.05 else 'does not survive'
    display(Markdown(
        f'**Headline:** FP16 GRPO gain = `{delta_fp16:+.3f}`, W4 GRPO gain = `{delta_w4:+.3f}`, '
        f'gain survival = `{gain_survival:+.3f}`. Under bnb-NF4 W4, the observed GRPO gain **{verdict}** quantization on this test-50 smoke.'
    ))
    display(Markdown(
        f'Base quantization change = `{quant_drop_base:+.3f}` in the saved convention `FP16 - W4`; '
        f'GRPO quantization change = `{quant_drop_g8:+.3f}`. Positive means W4 reduced accuracy relative to FP16.'
    ))

if not matrix.empty:
    for _, row in matrix.iterrows():
        display(Markdown(
            f"- `{row['label']}`: {int(row['correct'])}/{int(row['num_examples'])} correct = "
            f"`{row['accuracy']:.3f}` +/- `{row['accuracy_stderr']:.3f}` (1 sigma)."
        ))

## 4. Plots

Accuracy is the direct task metric. Reference PPL and completion length are diagnostics: they help identify whether quantization changes model behavior even when exact-match accuracy is noisy.

In [ ]:
if matrix.empty:
    print('No matrix loaded; skipping plots.')
else:
    labels = matrix['label'].tolist()
    colors = ['tab:gray' if v == 'base' else 'tab:blue' for v in matrix['variant']]
    hatches = ['' if p == 'fp16' else '//' for p in matrix['precision']]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    bars = axes[0].bar(labels, matrix['accuracy'], yerr=matrix['accuracy_stderr'], capsize=4, color=colors)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    axes[0].set_title('GSM8K exact-match accuracy')
    axes[0].set_ylabel('accuracy')
    axes[0].set_ylim(0, max(0.3, float(matrix['accuracy'].max()) + 0.08))
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].grid(axis='y', alpha=0.25)

    bars = axes[1].bar(labels, matrix['reference_ppl'], color=colors)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    axes[1].set_title('Reference solution PPL')
    axes[1].set_ylabel('PPL, lower is better')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].grid(axis='y', alpha=0.25)

    bars = axes[2].bar(labels, matrix['completion_chars_mean'], color=colors)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    axes[2].set_title('Mean completion length')
    axes[2].set_ylabel('characters')
    axes[2].tick_params(axis='x', rotation=30)
    axes[2].grid(axis='y', alpha=0.25)

    fig.tight_layout()
    plt.show()

## 5. Quantization Effect By Checkpoint

`paired_quant_effect.csv` compares FP16 vs W4 for the same checkpoint on each example. This shows whether quantization broke or fixed individual examples, not just the aggregate accuracy.

In [ ]:
paired = pd.read_csv(PAIRED_QUANT) if PAIRED_QUANT.exists() else pd.DataFrame()
safe_display(paired.head(), 'No paired quantization file loaded.')

if not paired.empty:
    counts = paired.groupby(['variant', 'quant_change']).size().unstack(fill_value=0)
    for col in ['improved', 'worsened', 'unchanged']:
        if col not in counts:
            counts[col] = 0
    counts = counts[['improved', 'worsened', 'unchanged']]
    display(counts)

    ax = counts[['improved', 'worsened']].plot(kind='bar', figsize=(7, 4), color=['tab:green', 'tab:red'])
    ax.set_title('Examples changed by W4 quantization')
    ax.set_ylabel('count')
    ax.grid(axis='y', alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    display(Markdown(
        'For this run, W4 helped the base model on more examples than it hurt, but hurt the GRPO checkpoint on more examples than it helped. '
        'That is exactly the pattern behind the negative gain-survival value.'
    ))

## 6. Inspect Changed Examples

This section lists examples where W4 changed correctness. Use it to check whether the aggregate result is caused by parser quirks, prompt leakage, or genuine answer changes.

In [ ]:
if paired.empty:
    print('No paired rows to inspect.')
else:
    changed = paired[paired['quant_change'] != 'unchanged'].copy()
    inspect_cols = [
        'variant', 'index', 'target_answer', 'fp16_answer', 'w4_answer',
        'fp16_correct', 'w4_correct', 'quant_change', 'question'
    ]
    safe_display(changed[inspect_cols].sort_values(['variant', 'quant_change', 'index']))

## 7. Prediction Drilldown

Set `EXAMPLE_INDEX` to inspect all four completions for one GSM8K item. This is useful when a row looks suspicious in the paired table.

In [ ]:
EXAMPLE_INDEX = 1

prediction_labels = ['base_fp16', 'g8_dr100_fp16', 'base_w4', 'g8_dr100_w4']
prediction_frames = {label: read_jsonl(EVAL_DIR / f'{label}_predictions.jsonl') for label in prediction_labels}

for label, frame in prediction_frames.items():
    if frame.empty or EXAMPLE_INDEX not in set(frame['index']):
        continue
    row = frame[frame['index'] == EXAMPLE_INDEX].iloc[0]
    display(Markdown(f"### {label}"))
    print('target:', row.get('target_answer'))
    print('parsed:', row.get('parsed_answer'))
    print('correct:', row.get('exact_match'))
    print('prompt_leak:', row.get('prompt_leak'))
    print('\nquestion:\n', row.get('question'))
    print('\ncompletion:\n', row.get('completion'))

## 8. Weight Diagnostics Context

The earlier weight/outlier diagnostic found almost identical weight-level W4 proxy error for base and GRPO. That means the behavioral W4 result is not explained by obvious global weight-distribution drift. Load that summary here if available.

In [ ]:
weight_summary_path = OUTLIER_DIR / 'weight_outlier_summary.json'
weight_summary = read_json(weight_summary_path)

weight_rows = []
for model_summary in weight_summary.get('models', []):
    weight_rows.append({
        'model': model_summary.get('model'),
        'num_weight_tensors': model_summary.get('num_weight_tensors'),
        'channel_outlier_frac_weighted': model_summary.get('channel_outlier_frac_weighted'),
        'abs_outlier_frac_weighted': model_summary.get('abs_outlier_frac_weighted'),
        'w4_relative_mse_weighted': model_summary.get('w4_relative_mse_weighted'),
        'w4_snr_db_mean': model_summary.get('w4_snr_db_mean'),
    })
safe_display(pd.DataFrame(weight_rows), 'No weight diagnostic summary loaded.')

## 9. Current Conclusion And Next Step

Current conclusion from test-50:

- GRPO improved FP16 exact-match accuracy from `0.06` to `0.22` on this split.
- Under bnb-NF4 W4, base scored `0.12` and GRPO scored `0.10`.
- The observed GRPO gain therefore did **not** survive this W4 scheme.
- Because `n=50` is noisy and this is NF4 rather than AWQ, this is a strong smoke result, not the final quantization claim.

Recommended next step: rerun the same 4-way eval at `test100` or inspect the changed examples above to decide whether the W4 regression is real answer degradation or mostly parser/format sensitivity.